In [ ]:
Assignment 1 — File Reads
Objective
Practice reading different file formats into Spark DataFrames.
Tasks
1. Read customers.csv
2. Read the same file with:
o header
o inferSchema
3. Read orders.csv using an explicitly defined schema.
4. Convert customers.csv to Parquet.
5. Read the Parquet file.
6. Read products.csv and display the schema.7. Read multiple CSV files from a directory.
8. Read multiple Parquet files from a directory.
9. Read JSON data containing customer information.
10.Compare the schemas obtained using:
o inferSchema
o user-defined schema.

1. Read customers.csv

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
spark=SparkSession.builder.appName('assignment').master('local[*]').getOrCreate()
customers=spark.read.csv("customers.csv")
customers.show(5)

+-----------+----------+---------+--------------------+----------+---------+-----------+-------+-----------------+----------------+-------------+
|        _c0|       _c1|      _c2|                 _c3|       _c4|      _c5|        _c6|    _c7|              _c8|             _c9|         _c10|
+-----------+----------+---------+--------------------+----------+---------+-----------+-------+-----------------+----------------+-------------+
|customer_id|first_name|last_name|               email|     phone|     city|      state|country|registration_date|customer_segment|date_of_birth|
|          1|     Rahul|   Sharma|rahul.sharma1@gma...|9876543201|Bengaluru|  Karnataka|  India|       2023-08-17|         Premium|   1988-07-23|
|          2|     Priya|    Reddy|priya.reddy2@gmai...|9876543202|Hyderabad|  Telangana|  India|       2024-07-17|        Standard|   1993-06-29|
|          3|      Arun|    Kumar|arun.kumar3@gmail...|9876543203|  Chennai| Tamil Nadu|  India|       2024-04-02|          

2. Read the same file with:
header=True,
inferSchema=True

In [15]:
customers=spark.read.csv("customers.csv",header=True,inferSchema=True)
customers.show()

+-----------+----------+---------+--------------------+----------+----------+--------------+-------+-----------------+----------------+-------------+
|customer_id|first_name|last_name|               email|     phone|      city|         state|country|registration_date|customer_segment|date_of_birth|
+-----------+----------+---------+--------------------+----------+----------+--------------+-------+-----------------+----------------+-------------+
|          1|     Rahul|   Sharma|rahul.sharma1@gma...|9876543201| Bengaluru|     Karnataka|  India|       2023-08-17|         Premium|   1988-07-23|
|          2|     Priya|    Reddy|priya.reddy2@gmai...|9876543202| Hyderabad|     Telangana|  India|       2024-07-17|        Standard|   1993-06-29|
|          3|      Arun|    Kumar|arun.kumar3@gmail...|9876543203|   Chennai|    Tamil Nadu|  India|       2024-04-02|           Basic|   1991-02-17|
|          4|     Sneha|    Patel|sneha.patel4@gmai...|9876543204|    Mumbai|   Maharashtra|  India|

3. Read orders.csv using an explicitly defined schema.

In [26]:
orders_scheme=StructType([
    StructField("order_id",LongType(),True),
    StructField("customer_id", IntegerType(),True),
    StructField("order_date", DateType(), True),
    StructField("order_status", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("shipping_city", StringType(), True),
    StructField("shipping_state", StringType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("discount_amount", DoubleType(), True),
    StructField("shipping_cost", DoubleType(), True)
    ])
                        

In [27]:
orders=spark.read.csv("orders.csv",header=True,schema=orders_scheme)
orders.show(5)
orders.printSchema()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|    1001|         29|2026-07-11|    Returned|          Card|         Pune|   Maharashtra|    10642.04|        1223.36|       173.61|
|    1002|         36|2025-11-03|   Completed|    NetBanking|       Mumbai|   Maharashtra|    13996.81|         207.17|        46.55|
|    1003|          7|2026-07-21|   Completed|    NetBanking|        Kochi|        Kerala|     2585.08|         218.51|        66.83|
|    1004|         41|2025-09-28|   Completed|          Card|    Bengaluru|     Karnataka|     7729.51|         502.24|       165.89|
|    1005|         18|2025-02-15|    Returned|    NetBanking| 

4. Convert customers.csv to Parquet.

In [30]:
customers=spark.read.csv("customers.csv",header=True,inferSchema=True)
customers.write.mode("overwrite").parquet("customers_parquet")


5. Read the Parquet file.

In [32]:
customers_parquet = spark.read.parquet("customers_parquet")
customers_parquet.show(5)
customers_parquet.printSchema()

+-----------+----------+---------+--------------------+----------+---------+-----------+-------+-----------------+----------------+-------------+
|customer_id|first_name|last_name|               email|     phone|     city|      state|country|registration_date|customer_segment|date_of_birth|
+-----------+----------+---------+--------------------+----------+---------+-----------+-------+-----------------+----------------+-------------+
|          1|     Rahul|   Sharma|rahul.sharma1@gma...|9876543201|Bengaluru|  Karnataka|  India|       2023-08-17|         Premium|   1988-07-23|
|          2|     Priya|    Reddy|priya.reddy2@gmai...|9876543202|Hyderabad|  Telangana|  India|       2024-07-17|        Standard|   1993-06-29|
|          3|      Arun|    Kumar|arun.kumar3@gmail...|9876543203|  Chennai| Tamil Nadu|  India|       2024-04-02|           Basic|   1991-02-17|
|          4|     Sneha|    Patel|sneha.patel4@gmai...|9876543204|   Mumbai|Maharashtra|  India|       2023-07-29|         P

6. Read products.csv and display the schema.

In [37]:
products=spark.read.csv("products.csv",header=True,inferSchema=True)
products.show(5)
products.printSchema()

+----------+--------------+-----------+------------+-----+-----------+---------+-------------+-----------+--------------+--------------+
|product_id|  product_name|   category|sub_category|brand|supplier_id|unit_cost|selling_price|launch_date|product_rating|stock_quantity|
+----------+--------------+-----------+------------+-----+-----------+---------+-------------+-----------+--------------+--------------+
|         1| Nova Mobile 1|Electronics|      Mobile| Nova|        101|  14238.0|     21486.98| 2024-03-28|           3.7|            70|
|         2| Apex Mobile 2|Electronics|      Mobile| Apex|        102|  16896.0|     22765.78| 2022-04-07|           4.7|            78|
|         3|Urban Mobile 3|Electronics|      Mobile|Urban|        103|  20760.0|     25202.49| 2025-10-25|           3.8|            32|
|         4|Prime Mobile 4|Electronics|      Mobile|Prime|        104|  12808.0|     16684.31| 2024-08-16|           4.1|           497|
|         5| Nova Laptop 1|Electronics|  

7. Read multiple CSV files from a directory.

In [43]:
data = spark.read.csv("ecommerce_datasets",header=True,inferSchema=True)
data.show(5)

+-------------+--------+----------+--------+----------+----------------+-----------+
|order_item_id|order_id|product_id|quantity|unit_price|discount_percent|item_amount|
+-------------+--------+----------+--------+----------+----------------+-----------+
|            1|    1001|        38|       2|  33226.07|            21.5|   52164.93|
|            2|    1002|        35|       4|   13518.2|           24.18|    40998.0|
|            3|    1002|        34|       3|  20157.95|            2.26|   59107.14|
|            4|    1002|         4|       1|  16684.31|            6.61|   15581.48|
|            5|    1003|         6|       3|  33895.24|           16.82|   84582.18|
+-------------+--------+----------+--------+----------+----------------+-----------+
only showing top 5 rows



8. Read multiple Parquet files from a directory.

In [42]:
df = spark.read.parquet("customers_parquet")

df.show(5)

+-----------+----------+---------+--------------------+----------+---------+-----------+-------+-----------------+----------------+-------------+
|customer_id|first_name|last_name|               email|     phone|     city|      state|country|registration_date|customer_segment|date_of_birth|
+-----------+----------+---------+--------------------+----------+---------+-----------+-------+-----------------+----------------+-------------+
|          1|     Rahul|   Sharma|rahul.sharma1@gma...|9876543201|Bengaluru|  Karnataka|  India|       2023-08-17|         Premium|   1988-07-23|
|          2|     Priya|    Reddy|priya.reddy2@gmai...|9876543202|Hyderabad|  Telangana|  India|       2024-07-17|        Standard|   1993-06-29|
|          3|      Arun|    Kumar|arun.kumar3@gmail...|9876543203|  Chennai| Tamil Nadu|  India|       2024-04-02|           Basic|   1991-02-17|
|          4|     Sneha|    Patel|sneha.patel4@gmai...|9876543204|   Mumbai|Maharashtra|  India|       2023-07-29|         P

In [50]:
data = [
    (1, "Rahul", "Bengaluru", "India"),
    (2, "Priya", "Hyderabad", "India"),
    (3, "Arun", "Chennai", "India")
]

columns = ["customer_id", "first_name", "city", "country"]

df = spark.createDataFrame(data, columns)

df.show()

df.write.mode("overwrite").json("customers_json")

+-----------+----------+---------+-------+
|customer_id|first_name|     city|country|
+-----------+----------+---------+-------+
|          1|     Rahul|Bengaluru|  India|
|          2|     Priya|Hyderabad|  India|
|          3|      Arun|  Chennai|  India|
+-----------+----------+---------+-------+



9. Read JSON data containing customer information.

In [54]:
customers_json = spark.read.json("customers_json")
customers_json.show()
customers_json.printSchema()

+---------+-------+-----------+----------+
|     city|country|customer_id|first_name|
+---------+-------+-----------+----------+
|Bengaluru|  India|          1|     Rahul|
|Hyderabad|  India|          2|     Priya|
|  Chennai|  India|          3|      Arun|
+---------+-------+-----------+----------+

root
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- first_name: string (nullable = true)



10.Compare the schemas obtained using:
o inferSchema
o user-defined schema

Using inferSchema

In [55]:
customers_infer = spark.read.csv("customers.csv",header=True,inferSchema=True)
customers_infer.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- date_of_birth: date (nullable = true)



Using user-defined schema

In [56]:
customer_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("registration_date", DateType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("date_of_birth", DateType(), True)
])

In [58]:
customers_schema = spark.read.csv("customers.csv",header=True,schema=customer_schema)
customers_schema.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- date_of_birth: date (nullable = true)



Assignment 2 — File Writes
Using orders DataFrame:
Tasks
1. Write orders into CSV.
2. Write orders into JSON.
3. Write orders into Parquet.
4. Write orders into ORC.
5. Write orders partitioned by order_status.
6. Write orders partitioned by:
o year
o month
7. Write completed orders into a separate Parquet location.
8. Write the output using:
o overwrite
o append
9. Compare the generated files.
10. Read the generated partitioned data back.


In [61]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import year, month

spark = SparkSession.builder.appName("Assignment 2 - File Writes").master("local[*]").getOrCreate()

orders = spark.read.csv("orders.csv",header=True,inferSchema=True)
orders.show(5)
orders.printSchema()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|    1001|         29|2026-07-11|    Returned|          Card|         Pune|   Maharashtra|    10642.04|        1223.36|       173.61|
|    1002|         36|2025-11-03|   Completed|    NetBanking|       Mumbai|   Maharashtra|    13996.81|         207.17|        46.55|
|    1003|          7|2026-07-21|   Completed|    NetBanking|        Kochi|        Kerala|     2585.08|         218.51|        66.83|
|    1004|         41|2025-09-28|   Completed|          Card|    Bengaluru|     Karnataka|     7729.51|         502.24|       165.89|
|    1005|         18|2025-02-15|    Returned|    NetBanking| 

1. Write orders into CSV.


In [63]:
orders.write.mode("overwrite").option("header", True).csv("output/orders_csv")


2. Write orders into JSON.


In [64]:
orders.write.mode("overwrite").json("output/orders_json")

3. Write orders into Parquet.


In [65]:
orders.write.mode("overwrite").parquet("output/orders_parquet")

4. Write orders into ORC.


In [66]:
orders.write.mode("overwrite").orc("output/orders_orc")

5. Write orders partitioned by order_status.


In [67]:
orders.write.mode("overwrite").partitionBy("order_status").parquet("output/orders_by_status")

6. Write orders partitioned by:
o year
o month

In [68]:
orders_with_date = orders.withColumn("year", year("order_date"))\
                            .withColumn("month", month("order_date"))

In [69]:
orders_with_date.write.mode("overwrite").partitionBy("year", "month").parquet("output/orders_by_year_month")

7. Write completed orders into a separate Parquet location.

In [70]:
completed_orders = orders.filter(orders.order_status == "Completed")

In [71]:
completed_orders.write.mode("overwrite").parquet("output/completed_orders")

In [73]:
completed_orders.show(5)

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|    1002|         36|2025-11-03|   Completed|    NetBanking|       Mumbai|   Maharashtra|    13996.81|         207.17|        46.55|
|    1003|          7|2026-07-21|   Completed|    NetBanking|        Kochi|        Kerala|     2585.08|         218.51|        66.83|
|    1004|         41|2025-09-28|   Completed|          Card|    Bengaluru|     Karnataka|     7729.51|         502.24|       165.89|
|    1006|         29|2026-07-19|   Completed|          Card|         Pune|   Maharashtra|    10673.31|        1121.81|       237.36|
|    1008|          3|2025-11-12|   Completed|    NetBanking| 

8. Write the output using:
o overwrite
o append


In [74]:
orders.write.mode("overwrite").parquet("output/orders_test")

In [75]:
orders.write.mode("append").parquet("output/orders_test")

9. Compare the generated files.


In [76]:
csv_df = spark.read.option("header", True).csv("output/orders_csv")

json_df = spark.read.json("output/orders_json")

parquet_df = spark.read.parquet("output/orders_parquet")

orc_df = spark.read.orc("output/orders_orc")

In [77]:
print("CSV:", csv_df.count())
print("JSON:", json_df.count())
print("Parquet:", parquet_df.count())
print("ORC:", orc_df.count())

CSV: 110
JSON: 110
Parquet: 110
ORC: 110


In [78]:
csv_df.printSchema()

json_df.printSchema()

parquet_df.printSchema()

orc_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- shipping_state: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- discount_amount: string (nullable = true)
 |-- shipping_cost: string (nullable = true)

root
 |-- customer_id: long (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- shipping_state: string (nullable = true)
 |-- total_amount: double (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: d

10. Read the generated partitioned data back.

In [85]:
orders_by_status = spark.read.parquet("output/orders_by_status")

orders_by_status.show(5)
orders_by_date = spark.read.parquet("output/orders_by_year_month")
orders_by_date.show(5)

+--------+-----------+----------+--------------+-------------+--------------+------------+---------------+-------------+------------+
|order_id|customer_id|order_date|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|order_status|
+--------+-----------+----------+--------------+-------------+--------------+------------+---------------+-------------+------------+
|    1002|         36|2025-11-03|    NetBanking|       Mumbai|   Maharashtra|    13996.81|         207.17|        46.55|   Completed|
|    1003|          7|2026-07-21|    NetBanking|        Kochi|        Kerala|     2585.08|         218.51|        66.83|   Completed|
|    1004|         41|2025-09-28|          Card|    Bengaluru|     Karnataka|     7729.51|         502.24|       165.89|   Completed|
|    1006|         29|2026-07-19|          Card|         Pune|   Maharashtra|    10673.31|        1121.81|       237.36|   Completed|
|    1008|          3|2025-11-12|    NetBanking|      Chennai|

Assignment 3 — withColumn()
Using orders:
Tasks
1. Create net_amount:
total_amount - discount_amount
2. Create final_amount:
net_amount + shipping_cost
3. Create discount_percentage.
4. Create order_year.
5. Create order_month.
6. Create order_quarter.
7. Create order_week.
8. Create is_high_value_order.
9. Create order_category:
< 500 → Low
500-2000 → Medium
> 2000 → High
10.Create order_age_days based on today's date

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Assignment 3 - withColumn").master("local[*]").getOrCreate()
orders = spark.read.csv("orders.csv",header=True,inferSchema=True)

# 1. net_amount
orders = orders.withColumn("net_amount",col("total_amount") - col("discount_amount"))

# 2. final_amount
orders = orders.withColumn("final_amount",col("net_amount") + col("shipping_cost"))

# 3. discount_percentage
orders = orders.withColumn("discount_percentage",(col("discount_amount") / col("total_amount")) * 100)

# 4. order_year
orders = orders.withColumn("order_year",year(col("order_date")))

# 5. order_month
orders = orders.withColumn("order_month",month(col("order_date")))

# 6. order_quarter
orders = orders.withColumn("order_quarter",quarter(col("order_date")))

# 7. order_week
orders = orders.withColumn("order_week",weekofyear(col("order_date")))

# 8. is_high_value_order
orders = orders.withColumn("is_high_value_order",when(col("final_amount") > 2000, True).otherwise(False))

# 9. order_category
orders = orders.withColumn("order_category",when(col("final_amount") < 500, "Low")
    .when(col("final_amount") <= 2000, "Medium").otherwise("High"))

# 10. order_age_days
orders = orders.withColumn("order_age_days",datediff(current_date(), col("order_date")))

orders.show(5,truncate=False)

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+------------------+------------------+-------------------+----------+-----------+-------------+----------+-------------------+--------------+--------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|net_amount        |final_amount      |discount_percentage|order_year|order_month|order_quarter|order_week|is_high_value_order|order_category|order_age_days|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+------------------+------------------+-------------------+----------+-----------+-------------+----------+-------------------+--------------+--------------+
|1001    |29         |2026-07-11|Returned    |Card          |Pune         |Maharashtra   |10642.04    |1223.36        |173.61     

Assignment 4 — withColumnRenamed()
Using customers:
Rename:
customer_id → cust_id
first_name → fname
last_name → lname
registration_date → signup_date
customer_segment → segment

In [95]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Assignment 4 - withColumnRenamed").master("local[*]").getOrCreate()

customers = spark.read.csv("customers.csv",header=True,inferSchema=True)

customers = customers.withColumnRenamed("customer_id", "cust_id")

customers = customers.withColumnRenamed("first_name", "fname")

customers = customers.withColumnRenamed("last_name", "lname")
customers = customers.withColumnRenamed("registration_date", "signup_date")

customers = customers.withColumnRenamed("customer_segment", "segment")

customers.show(5)
customers.printSchema(5)


+-------+------+------+--------------------+----------+---------+-----------+-------+-----------+--------+-------------+
|cust_id| fname| lname|               email|     phone|     city|      state|country|signup_date| segment|date_of_birth|
+-------+------+------+--------------------+----------+---------+-----------+-------+-----------+--------+-------------+
|      1| Rahul|Sharma|rahul.sharma1@gma...|9876543201|Bengaluru|  Karnataka|  India| 2023-08-17| Premium|   1988-07-23|
|      2| Priya| Reddy|priya.reddy2@gmai...|9876543202|Hyderabad|  Telangana|  India| 2024-07-17|Standard|   1993-06-29|
|      3|  Arun| Kumar|arun.kumar3@gmail...|9876543203|  Chennai| Tamil Nadu|  India| 2024-04-02|   Basic|   1991-02-17|
|      4| Sneha| Patel|sneha.patel4@gmai...|9876543204|   Mumbai|Maharashtra|  India| 2023-07-29| Premium|   2000-03-25|
|      5|Vikram|   Das|vikram.das5@gmail...|9876543205|     Pune|Maharashtra|  India| 2023-06-28|Standard|   2001-03-30|
+-------+------+------+---------

Assignment 5 — Filters
Using orders:
Tasks
Find:
1. Orders greater than ₹5,000.
2. Completed orders.
3. Cancelled orders.
4. Orders from Karnataka.
5. Orders from Karnataka or Maharashtra.
6. Orders between ₹1,000 and ₹10,000.
7. Orders placed during 2025.
8. Orders where discount is greater than 20%.
9. Orders where payment method is UPI.
10. Orders where status is not Cancelled.
11. Orders where shipping city is NULL.
12. Orders where amount is NOT NULL.
13. Orders satisfying multiple conditions.
14. Find the top-value completed orders where:
total_amount > 5000
AND discount_amount > 500
AND payment_method = 'UPI'

In [99]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year

spark = SparkSession.builder \
    .appName("Assignment 5 - Filters") \
    .master("local[*]") \
    .getOrCreate()

orders = spark.read.csv("orders.csv",header=True,inferSchema=True)

# 1. Orders greater than ₹5,000
result1 = orders.filter(col("total_amount") > 5000)
result1.show(5)


# 2. Completed orders
result2 = orders.filter(col("order_status") == "Completed")
result2.show(5)


# 3. Cancelled orders
result3 = orders.filter(col("order_status") == "Cancelled")
result3.show(5)


# 4. Orders from Karnataka
result4 = orders.filter(col("shipping_state") == "Karnataka")
result4.show(5)


# 5. Orders from Karnataka or Maharashtra
result5 = orders.filter((col("shipping_state") == "Karnataka") |(col("shipping_state") == "Maharashtra"))
result5.show(5)


# 6. Orders between ₹1,000 and ₹10,000
result6 = orders.filter((col("total_amount") >= 1000) &(col("total_amount") <= 10000))
result6.show(5)


# 7. Orders placed during 2025
result7 = orders.filter(year(col("order_date")) == 2025)
result7.show(5)


# 8. Orders where discount is greater than 20%
result8 = orders.filter((col("discount_amount") / col("total_amount")) * 100 > 20)
result8.show(5)


# 9. Orders where payment method is UPI
result9 = orders.filter(col("payment_method") == "UPI")
result9.show(5)


# 10. Orders where status is not Cancelled
result10 = orders.filter(col("order_status") != "Cancelled")
result10.show(5)


# 11. Orders where shipping city is NULL
result11 = orders.filter(col("shipping_city").isNull())
result11.show(5)


# 12. Orders where amount is NOT NULL
result12 = orders.filter(col("total_amount").isNotNull())
result12.show(5)


# 13. Orders satisfying multiple conditions
result13 = orders.filter((col("total_amount") > 5000) &(col("order_status") == "Completed") &(col("payment_method") == "UPI"))
result13.show(5)


# 14. Top-value completed orders
# Conditions:
# total_amount > 5000
# discount_amount > 500
# payment_method = UPI

result14 = orders.filter((col("total_amount") > 5000) &(col("order_status") == "Completed") &
    (col("discount_amount") > 500) &(col("payment_method") == "UPI")).orderBy(col("total_amount").desc())

result14.show(5)


+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|    1001|         29|2026-07-11|    Returned|          Card|         Pune|   Maharashtra|    10642.04|        1223.36|       173.61|
|    1002|         36|2025-11-03|   Completed|    NetBanking|       Mumbai|   Maharashtra|    13996.81|         207.17|        46.55|
|    1004|         41|2025-09-28|   Completed|          Card|    Bengaluru|     Karnataka|     7729.51|         502.24|       165.89|
|    1006|         29|2026-07-19|   Completed|          Card|         Pune|   Maharashtra|    10673.31|        1121.81|       237.36|
|    1007|         45|2025-06-02|     Pending|           UPI| 

Assignment 6 — Aggregation Functions
Using orders:
Tasks
Calculate:
1. Total number of orders.
2. Total revenue.
3. Average order value.
4. Minimum order value.
5. Maximum order value.
6. Total discount.
7. Total shipping cost.
8. Number of orders by status.
9. Revenue by state.
10. Revenue by payment method.
11. Average order value by customer.
12.Maximum order value by customer.
13.Monthly revenue.
14.Monthly order count.
15. Revenue by state and payment method

In [101]:
spark = SparkSession.builder.appName("Assignment 6 - Aggregation Functions").master("local[*]").getOrCreate()

orders = spark.read.csv("orders.csv",header=True,inferSchema=True)

# 1. Total number of orders
total_orders = orders.agg(count("order_id").alias("total_orders"))
total_orders.show()


# 2. Total revenue
total_revenue = orders.agg(sum("total_amount").alias("total_revenue"))
total_revenue.show()


# 3. Average order value
average_order = orders.agg(avg("total_amount").alias("average_order_value"))
average_order.show()


# 4. Minimum order value
minimum_order = orders.agg(min("total_amount").alias("minimum_order_value"))
minimum_order.show()


# 5. Maximum order value
maximum_order = orders.agg(max("total_amount").alias("maximum_order_value"))
maximum_order.show()


# 6. Total discount
total_discount = orders.agg(sum("discount_amount").alias("total_discount"))
total_discount.show()


# 7. Total shipping cost
total_shipping = orders.agg(sum("shipping_cost").alias("total_shipping_cost"))
total_shipping.show()


# 8. Number of orders by status
orders_by_status = orders.groupBy("order_status").agg(count("order_id").alias("order_count"))
orders_by_status.show()


# 9. Revenue by state
revenue_by_state = orders.groupBy("shipping_state").agg(sum("total_amount").alias("revenue"))

revenue_by_state.show()


# 10. Revenue by payment method
revenue_by_payment = orders.groupBy("payment_method").agg(sum("total_amount").alias("revenue"))
revenue_by_payment.show()


# 11. Average order value by customer
average_by_customer = orders.groupBy("customer_id").agg(avg("total_amount").alias("average_order_value"))
average_by_customer.show()


# 12. Maximum order value by customer
maximum_by_customer = orders.groupBy("customer_id").agg(max("total_amount").alias("maximum_order_value"))
maximum_by_customer.show()


# 13. Monthly revenue
monthly_revenue = orders.groupBy(year("order_date").alias("year"),month("order_date").alias("month")).agg(sum("total_amount").alias("monthly_revenue")
).orderBy("year","month")
monthly_revenue.show()


# 14. Monthly order count
monthly_order_count = orders.groupBy(year("order_date").alias("year"),month("order_date").alias("month")).agg(count("order_id").alias("order_count")
).orderBy("year","month")
monthly_order_count.show()

# 15. Revenue by state and payment method
revenue_state_payment = orders.groupBy("shipping_state","payment_method").agg(sum("total_amount").alias("revenue"))
revenue_state_payment.show()

+------------+
|total_orders|
+------------+
|         110|
+------------+

+-----------------+
|    total_revenue|
+-----------------+
|908569.1600000005|
+-----------------+

+-------------------+
|average_order_value|
+-------------------+
|  8259.719636363641|
+-------------------+

+-------------------+
|minimum_order_value|
+-------------------+
|             352.12|
+-------------------+

+-------------------+
|maximum_order_value|
+-------------------+
|           14983.86|
+-------------------+

+-----------------+
|   total_discount|
+-----------------+
|98682.76000000002|
+-----------------+

+-------------------+
|total_shipping_cost|
+-------------------+
| 15178.699999999999|
+-------------------+

+------------+-----------+
|order_status|order_count|
+------------+-----------+
|    Returned|         19|
|   Completed|         51|
|   Cancelled|         26|
|     Pending|         14|
+------------+-----------+

+--------------+------------------+
|shipping_state|         

Assignment 7 — Window Functions
Using orders:
Tasks
1. Find the customer's first order.
2. Find the customer's latest order.
3. Rank customers based on total spending.
4. Rank orders within each customer.
5. Find the top 3 orders for every customer.
6. Calculate running revenue for each customer.
7. Calculate previous order amount using lag().
8. Calculate next order amount using lead().
9. Calculate difference between current and previous order.
10.Calculate cumulative order count for each customer.
Bonus
Find customers whose latest order value is greater than their previous order.

In [4]:
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Assignment 7 - Window Functions") \
    .master("local[*]") \
    .getOrCreate()

orders = spark.read.csv("orders.csv",header=True,inferSchema=True)

# Window for each customer, ordered by order date
customer_window = Window \
    .partitionBy("customer_id") \
    .orderBy("order_date")


# 1. Find the customer's first order
first_order = orders.withColumn("row_num",row_number().over(customer_window)).filter(col("row_num") == 1)
first_order.show(5)


# 2. Find the customer's latest order
latest_order = orders.withColumn("row_num",row_number().over(Window.partitionBy("customer_id").orderBy(col("order_date").desc()))
).filter(col("row_num") == 1)

latest_order.show(5)


# 3. Rank customers based on total spending
customer_spending = orders.groupBy("customer_id"
).agg(sum("total_amount").alias("total_spending"))

spending_window = Window.orderBy(col("total_spending").desc())

ranked_customers = customer_spending.withColumn("spending_rank",rank().over(spending_window))

ranked_customers.show(5)


# 4. Rank orders within each customer
ranked_orders = orders.withColumn("order_rank",rank().over(customer_window))

ranked_orders.show(5)


# 5. Find top 3 orders for every customer
top_3_orders = orders.withColumn("order_rank",row_number().over(Window.partitionBy("customer_id").orderBy(col("total_amount").desc()))
).filter(col("order_rank") <= 3)

top_3_orders.show(5)


# 6. Calculate running revenue for each customer
running_revenue = orders.withColumn("running_revenue",sum("total_amount").over(customer_window.rowsBetween(Window.unboundedPreceding,
Window.currentRow)))
running_revenue.show(5)


# 7. Previous order amount using lag()
previous_order = orders.withColumn("previous_order_amount",lag("total_amount").over(customer_window))

previous_order.show(5)


# 8. Next order amount using lead()
next_order = orders.withColumn("next_order_amount",lead("total_amount").over(customer_window))
next_order.show(5)


# 9. Difference between current and previous order
order_difference = orders.withColumn("previous_order_amount",lag("total_amount").over(customer_window)
).withColumn("order_difference",col("total_amount") - col("previous_order_amount"))
order_difference.show(5)


# 10. Cumulative order count for each customer
cumulative_count = orders.withColumn("cumulative_order_count",count("order_id").over(
        customer_window.rowsBetween(Window.unboundedPreceding,Window.currentRow)))
cumulative_count.show(5)


# BONUS
# Customers whose latest order value is greater than their previous order

bonus_window = Window.partitionBy("customer_id").orderBy(col("order_date").desc())
bonus = orders.withColumn("row_num",row_number().over(bonus_window)
).withColumn("previous_order_amount",lead("total_amount").over(bonus_window)
).filter((col("row_num") == 1) &(col("total_amount") > col("previous_order_amount")))

bonus.show(5)

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+-------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|row_num|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+-------+
|    1022|          1|2025-11-08|     Pending|    NetBanking|    Bengaluru|     Karnataka|     4517.63|         582.62|       126.95|      1|
|    1056|          2|2025-04-06|   Completed|    NetBanking|    Hyderabad|     Telangana|     4625.26|         561.14|       172.71|      1|
|    1092|          3|2025-05-10|   Completed|          Cash|      Chennai|    Tamil Nadu|     7683.68|        1180.36|        42.93|      1|
|    1098|          4|2025-10-28|   Completed|          Cash|       Mumbai|   Maharashtra|     5569.41|          719.6|       150.16|      1|
|    1

Assignment 8 — Joins
Use:
customers
orders
order_items
products
payments
Tasks
Implement and understand:
1. Inner join
2. Left join
3. Right join
4. Full outer join
5. Left semi join
6. Left anti join
7. Cross join
8. Self join
Questions
1. Find customers who placed orders.
2. Find customers who never placed an order.
3. Find products that were never sold.
4. Find orders without payment records.
5. Find customers with successful payments.
6. Find all customers and their orders.
7. Find employees and their managers using self join.
8. Create an order-product dataset by joining:
o orders
o order_items
o products.


In [ ]:
customers=spark.read.csv("customers.csv",header=True,inferSchema=True)
orders=spark.read.csv("orders.csv",header=True,inferSchema=True)
order_items=spark.read.csv("order_items.csv",header=True,inferSchema=True)
payments =spark.read.csv("payments.csv",header=True,inferSchema=True)
employees =spark.read.csv("employees.csv",header=True,inferSchema=True)
products = spark.read.csv("products.csv",header=True,inferSchema=True)
orders.show(5)
#1. inner join
customer_orders_inner=customers.join(orders,customers.customer_id==orders.customer_id)
customer_orders_inner.show(5)
# 2. Left Join
customer_orders_left = customers.join(orders,customers.customer_id == orders.customer_id,"left")
customer_orders_left.show(5)


# 3. Right Join
customer_orders_right = customers.join(orders,customers.customer_id == orders.customer_id,"right")
customer_orders_right.show(5)


# 4. Full Outer Join
customer_orders_full = customers.join(orders,customers.customer_id == orders.customer_id,"full")

customer_orders_full.show(5)


# 5. Left Semi Join
customers_with_orders = customers.join(orders,customers.customer_id == orders.customer_id,"left_semi")
customers_with_orders.show(5)


# 6. Left Anti Join
customers_without_orders = customers.join(orders,customers.customer_id == orders.customer_id,"left_anti"
)
customers_without_orders.show(5)


# 7. Cross Join
customer_product_cross = customers.crossJoin(products)
customer_product_cross.show(5)


# 8. Self Join
employees_alias = employees.alias("e")
managers_alias = employees.alias("m")

employee_managers = employees_alias.join(managers_alias,col("e.manager_id") == col("m.employee_id"),"left").select(
    col("e.employee_id"),
    col("e.employee_name"),
    col("e.manager_id"),
    col("m.employee_name").alias("manager_name")
)

employee_managers.show(5)


# Question 1
# Find customers who placed orders
customers_with_orders = customers.join(orders,customers.customer_id == orders.customer_id,"left_semi")

customers_with_orders.show(5)


# Question 2
# Find customers who never placed an order
customers_without_orders = customers.join(orders,customers.customer_id == orders.customer_id,"left_anti")

customers_without_orders.show()


# Question 3
# Find products that were never sold
sold_products = order_items.select("product_id").distinct()

products_never_sold = products.join(sold_products,products.product_id == sold_products.product_id,"left_anti")

products_never_sold.show()


# Question 4
# Find orders without payment records
orders_without_payment = orders.join(payments,orders.order_id == payments.order_id,"left_anti")

orders_without_payment.show()


# Question 5
# Find customers with successful payments
successful_payments = payments.filter(col("payment_status") == "Success"


customers_successful_payment = customers.join(orders,customers.customer_id == orders.customer_id,"inner").join(
    successful_payments,orders.order_id == successful_payments.order_id, "inner").select(
    customers.customer_id,
    customers.first_name,
    customers.last_name).distinct()
customers_successful_payment.show()


# Question 6
# Find all customers and their orders
all_customers_orders = customers.join(orders,customers.customer_id == orders.customer_id,"left")

all_customers_orders.show()


# Question 7
# Find employees and their managers
employee_managers.show()


# Question 8
# Create order-product dataset
order_product = orders.join(order_items,orders.order_id == order_items.order_id,"inner").join(
    products, order_items.product_id == products.product_id,"inner")

order_product.show()


spark.stop()


+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|    1001|         29|2026-07-11|    Returned|          Card|         Pune|   Maharashtra|    10642.04|        1223.36|       173.61|
|    1002|         36|2025-11-03|   Completed|    NetBanking|       Mumbai|   Maharashtra|    13996.81|         207.17|        46.55|
|    1003|          7|2026-07-21|   Completed|    NetBanking|        Kochi|        Kerala|     2585.08|         218.51|        66.83|
|    1004|         41|2025-09-28|   Completed|          Card|    Bengaluru|     Karnataka|     7729.51|         502.24|       165.89|
|    1005|         18|2025-02-15|    Returned|    NetBanking| 

Assignment 9 — orderBy()
Using orders:
Tasks
1. Sort orders by amount ascending.
2. Sort orders by amount descending.
3. Sort by date descending.
4. Sort by state and amount.
5. Sort by status ascending and amount descending.
6. Sort NULL values first.
7. Sort NULL values last.
8. Find the top 100 highest-value orders.
9. Find the 100 oldest orders.
10. Sort customers based on total spending.

In [20]:
spark = SparkSession.builder.appName("Assignment 9 - orderBy").master("local[*]") .getOrCreate()

orders = spark.read.csv("orders.csv",header=True,inferSchema=True)

# 1. Sort orders by amount ascending
result1=orders.orderBy(col("total_amount").asc())
result1.show(5)

# 2. Sort orders by amount descending
result2 = orders.orderBy(col("total_amount").desc())
result2.show(5)


# 3. Sort by date descending
result3 = orders.orderBy(col("order_date").desc())
result3.show(5)


# 4. Sort by state and amount
result4 = orders.orderBy(col("shipping_state").asc(),col("total_amount").asc())

result4.show(5)


# 5. Sort by status ascending and amount descending
result5 = orders.orderBy(col("order_status").asc(),col("total_amount").desc())
result5.show(5)


# 6. Sort NULL values first
result6 = orders.orderBy(col("shipping_city").asc_nulls_first())

result6.show(5)


# 7. Sort NULL values last
result7 = orders.orderBy(col("shipping_city").asc_nulls_last())

result7.show(5)


# 8. Find the top 100 highest-value orders
result8 = orders.orderBy(col("total_amount").desc()).limit(100)
result8.show(5)


# 9. Find the 100 oldest orders
result9 = orders.orderBy(col("order_date").asc()).limit(100)

result9.show(5)


# 10. Sort customers based on total spending
customer_spending = orders.groupBy("customer_id").agg(sum("total_amount").alias("total_spending"))
result10 = customer_spending.orderBy(col("total_spending").desc())

result10.show(5)


+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|    1005|         18|2025-02-15|    Returned|    NetBanking|    Hyderabad|     Telangana|      352.12|          81.46|        57.16|
|    1061|         12|2025-11-06|   Completed|          Card|       Mumbai|   Maharashtra|      507.98|           81.4|       244.09|
|    1041|          6|2026-04-27|   Completed|           UPI|        Delhi|         Delhi|      584.93|          94.66|       169.53|
|    1072|         10|2025-03-15|     Pending|    NetBanking|    Hyderabad|     Telangana|     1176.68|          279.8|       121.01|
|    1048|         28|2026-01-15|   Completed|          Cash| 

Assignment 10 — Date Functions
Using orders and customers:
Questions
1. Extract year from order date.
2. Extract month.
3. Find orders placed on weekends.
4. Find orders placed on Monday.
5. Calculate order age.
6. Find customers registered more than 1 year ago.
7. Calculate months since registration.
8. Find orders from the last 30 days.
9. Find month-end orders.
10. Generate YYYY-MM from order date.


In [2]:
spark = SparkSession.builder.appName("Assignment 10 - Date Functions").master("local[*]").getOrCreate()

orders = spark.read.csv("orders.csv",header=True,inferSchema=True)

customers = spark.read.csv("customers.csv",header=True,inferSchema=True)

# 1. Extract year from order date
result1 =orders.withColumn("order_year",year(col("order_date")))
result1.show(5)

# 2. Extract month.
result2=orders.withColumn("order_month",month(col("order_date")))
result2.show(5)

# 3. Find orders placed on Weekends.
result3=orders.filter(dayofweek(col("order_date")).isin(1,7))
result3.show(5)

# 4. Find orders placed on Monday.
result4=orders.filter(dayofweek(col("order_date"))==2)
result4.show(5)

# 5. Calculate order age
result5=orders.withColumn("Order_age",date_diff(current_date(),col("order_date")))
result5.show(5)

# 6. Find customers registered more than 1 year ago
result6 = customers.filter(col("registration_date") < add_months(current_date(), -12))
result6.show()

# 7. Calculate months since registration
result7=customers.withColumn("months_since_registration",(datediff(current_date(),col("registration_date"))/30).cast("int"))
result7.show(5)

# 8. Find orders from the last 30 days
result8=orders.filter(col("order_date")>=date_sub(current_date(),30))
result8.show(5)

# 9. Find month-end orders
result9 = orders.filter(col("order_date") == last_day(col("order_date")))
result9.show(5)

# 10. Generate YYYY-MM from order date
result10 = orders.withColumn("order_year_month",date_format(col("order_date"), "yyyy-MM"))
result10.show(5)



+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+----------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|order_year|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+----------+
|    1001|         29|2026-07-11|    Returned|          Card|         Pune|   Maharashtra|    10642.04|        1223.36|       173.61|      2026|
|    1002|         36|2025-11-03|   Completed|    NetBanking|       Mumbai|   Maharashtra|    13996.81|         207.17|        46.55|      2025|
|    1003|          7|2026-07-21|   Completed|    NetBanking|        Kochi|        Kerala|     2585.08|         218.51|        66.83|      2026|
|    1004|         41|2025-09-28|   Completed|          Card|    Bengaluru|     Karnataka|     7729.51|         502.24|       165.

Assignment 11 — String Functions
Using customers and products:
Questions
1. Convert names to uppercase.
2. Convert names to lowercase.
3. Create full name.
4. Find length of customer names.
5. Extract email username.
6. Extract email domain.
7. Extract phone country code.
8. Remove special characters from phone numbers.
9. Extract product brand from product description.
10. Standardize city names.
Example
From:
charan@gmail.com
extract:
gmail.com
using regexp_extract().

In [8]:
spark = SparkSession.builder.appName("Assignment 11 - String Functions").master("local[*]").getOrCreate()

customers = spark.read.csv("customers.csv",header=True,inferSchema=True)

products = spark.read.csv("products.csv",header=True,inferSchema=True)

# 1. Convert names to uppercase
result1 = customers.withColumn("first_name_upper",upper("first_name"))\
                    .withColumn("last_name_upper",upper("last_name"))

result1.show(5)

# 2. Convert names to lowercase
result2 = customers.withColumn("first_name_lower",lower("first_name"))\
                    .withColumn("last_name_lower",lower("last_name"))

result2.show(5)

# 3. Create full name
result3 = customers.withColumn("full_name",concat_ws(" ", "first_name", "last_name"))
result3.show(5)

# 4. Find length of customer names
result4 = customers.withColumn("first_name_length",length("first_name"))\
                    .withColumn("last_name_length",length("last_name"))
result4.show(5)

# 5. Extract email username
result5 = customers.withColumn("email_username",regexp_extract("email", r"^([^@]+)", 1))
result5.show(5)

# 6. Extract email domain
result6 = customers.withColumn("email_domain",regexp_extract("email", r"@(.+)$", 1))
result6.show(5)

# 7. Extract phone country code
result7 = customers.withColumn("country_code",regexp_extract("phone", r"^(\+\d+)", 1))
result7.show(5)

# 8. Remove special characters from phone numbers
result8 = customers.withColumn("clean_phone",regexp_replace("phone", r"[^0-9]", ""))
result8.show(5)

# 9. Extract product brand from product description
result9 = products.withColumn("product_brand",regexp_extract("product_name", r"^(\w+)", 1))
result9.show(5)

# 10. Standardize city names
result10 = customers.withColumn("standardized_city",initcap(trim("city")))
result10.show(5)


+-----------+----------+---------+--------------------+----------+---------+-----------+-------+-----------------+----------------+-------------+----------------+---------------+
|customer_id|first_name|last_name|               email|     phone|     city|      state|country|registration_date|customer_segment|date_of_birth|first_name_upper|last_name_upper|
+-----------+----------+---------+--------------------+----------+---------+-----------+-------+-----------------+----------------+-------------+----------------+---------------+
|          1|     Rahul|   Sharma|rahul.sharma1@gma...|9876543201|Bengaluru|  Karnataka|  India|       2023-08-17|         Premium|   1988-07-23|           RAHUL|         SHARMA|
|          2|     Priya|    Reddy|priya.reddy2@gmai...|9876543202|Hyderabad|  Telangana|  India|       2024-07-17|        Standard|   1993-06-29|           PRIYA|          REDDY|
|          3|      Arun|    Kumar|arun.kumar3@gmail...|9876543203|  Chennai| Tamil Nadu|  India|       20

Assignment 12 — Other Built-in PySpark Functions
Create exercises around:
Null handling
Questions
1. Replace NULL discount with 0.
2. Replace NULL city with "Unknown".
3. Create customer category based on spending.
4. Round revenue to 2 decimal places.
5. Calculate profit.
6. Extract values from an array.
7. Explode product arrays.
8. Find array length.
9. Check whether an array contains a specific value

In [10]:
spark = SparkSession.builder .appName("Assignment 12 - Built-in Functions").master("local[*]").getOrCreate()

orders = spark.read.csv("orders.csv",header=True,inferSchema=True)
customers = spark.read.csv("customers.csv",header=True,inferSchema=True)
products = spark.read.csv("products.csv",header=True,inferSchema=True)

# 1. Replace NULL discount with 0
result1 = orders.withColumn("discount_amount",coalesce(col("discount_amount"), lit(0)))
result1.show(5)

# 2. Replace NULL city with "Unknown"
result2 = customers.withColumn("city",coalesce(col("city"), lit("Unknown")))
result2.show(5)

# 3. Create customer category based on spending
customer_spending = orders.groupBy("customer_id").agg(sum("total_amount").alias("total_spending"))

result3 = customer_spending.withColumn("customer_category", when(col("total_spending") < 5000, "Low")
                                                            .when(col("total_spending") <= 15000, "Medium").otherwise("High"))
result3.show(5)

# 4. Round revenue to 2 decimal places
result4 = orders.withColumn("rounded_revenue",round(col("total_amount"), 2))
result4.show(5)

# 5. Calculate profit
result5 = products.withColumn("profit",col("selling_price") - col("unit_cost"))
result5.show(5)

# 6. Extract values from an array
array_df = products.withColumn("product_array",array(col("product_name"),col("brand"),col("category")))

result6 = array_df.select("product_name","product_array",
    col("product_array")[0].alias("first_value"),
    col("product_array")[1].alias("second_value"),
    col("product_array")[2].alias("third_value"))
result6.show(5,truncate=False)

# 7. Explode product arrays
result7 = array_df.select("product_name",explode("product_array").alias("product_value"))
result7.show(5,truncate=False)

# 8. Find array length
result8 = array_df.withColumn("array_length",size("product_array"))
result8.show(5,truncate=False)

# 9. Check whether an array contains a specific value
result9 = array_df.withColumn("contains_electronics",array_contains("product_array", "Electronics"))

result9.show(5,truncate=False)


+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|    1001|         29|2026-07-11|    Returned|          Card|         Pune|   Maharashtra|    10642.04|        1223.36|       173.61|
|    1002|         36|2025-11-03|   Completed|    NetBanking|       Mumbai|   Maharashtra|    13996.81|         207.17|        46.55|
|    1003|          7|2026-07-21|   Completed|    NetBanking|        Kochi|        Kerala|     2585.08|         218.51|        66.83|
|    1004|         41|2025-09-28|   Completed|          Card|    Bengaluru|     Karnataka|     7729.51|         502.24|       165.89|
|    1005|         18|2025-02-15|    Returned|    NetBanking| 

Assignment 13 — UDF
Create a Python UDF that accepts:
country_code
and returns:
country_name
Example:
IN → India
US → United States
UK → United Kingdom
AU → Australia
SG → Singapore
Tasks
1. Create a normal Python function.
2. Convert it into a PySpark UDF.
3. Apply it to a DataFrame.
4. Create a UDF that categorizes customers based on age.
5. Create a UDF that categorizes orders based on amount.
6. Create a UDF that masks phone numbers.
Bonus
Implement the same transformation using native PySpark functions and compare it
conceptually with the UDF approach.

In [15]:
from pyspark.sql.types import StringType

spark = SparkSession.builder.appName("Assignment 13 - UDF").master("local[*]").getOrCreate()

customers = spark.read.csv("customers.csv",header=True,inferSchema=True)

orders = spark.read.csv("orders.csv",header=True,inferSchema=True)

# 1. Normal Python function
def get_country_name(country_code):
    country_map = {
        "IN": "India",
        "US": "United States",
        "UK": "United Kingdom",
        "AU": "Australia",
        "SG": "Singapore"}
    return country_map.get(country_code, "Unknown")


# 2. Convert Python function into PySpark UDF
country_udf = udf(get_country_name, StringType())
# 3. Apply UDF to DataFrame
# Your customers.csv has a country column.
# Here we create country_code from the country value for demonstration.

def get_country_code(country):
    country_map = {
        "India": "IN",
        "United States": "US",
        "United Kingdom": "UK",
        "Australia": "AU",
        "Singapore": "SG"}
    return country_map.get(country, "Unknown")
country_code_udf = udf(get_country_code, StringType())
customers_with_code = customers.withColumn(
    "country_code",
    country_code_udf(col("country")))
result3 = customers_with_code.withColumn("country_name",country_udf(col("country_code")))

result3.select("customer_id","country","country_code","country_name").show()
# 4. UDF to categorize customers based on age

def age_category(age):
    if age is None:
        return "Unknown"
    elif age < 18:
        return "Minor"
    elif age < 30:
        return "Young"
    elif age < 50:
        return "Middle Age"
    else:
        return "Senior"


age_udf = udf(age_category, StringType())

customers_with_age = customers.withColumn("age",
    (datediff(current_date(), col("date_of_birth")) / 365.25).cast("int"))
result4 = customers_with_age.withColumn("age_category",age_udf(col("age")))

result4.select("customer_id","date_of_birth","age","age_category").show()


# 5. UDF to categorize orders based on amount

def order_category(amount):
    if amount is None:
        return "Unknown"
    elif amount < 500:
        return "Low"
    elif amount <= 2000:
        return "Medium"
    else:
        return "High"


order_category_udf = udf(order_category, StringType())

result5 = orders.withColumn("amount_category", order_category_udf(col("total_amount")))
result5.select("order_id","total_amount","amount_category").show()


# 6. UDF to mask phone numbers

def mask_phone(phone):
    if phone is None:
        return None
    digits = "".join(ch for ch in phone if ch.isdigit())
    if len(digits) <= 4:
        return "*" * len(digits)
    return "*" * (len(digits) - 4) + digits[-4:]
mask_phone_udf = udf(mask_phone, StringType())
result6 = customers.withColumn("masked_phone",mask_phone_udf(col("phone").cast("String")))
result6.select("customer_id","phone","masked_phone").show()

# Bonus - Native PySpark approach for order category

native_result = orders.withColumn("amount_category",
    when(col("total_amount").isNull(), "Unknown")
    .when(col("total_amount") < 500, "Low")
    .when(col("total_amount") <= 2000, "Medium")
    .otherwise("High")
)

native_result.select("order_id","total_amount","amount_category").show()

+-----------+-------+------------+------------+
|customer_id|country|country_code|country_name|
+-----------+-------+------------+------------+
|          1|  India|          IN|       India|
|          2|  India|          IN|       India|
|          3|  India|          IN|       India|
|          4|  India|          IN|       India|
|          5|  India|          IN|       India|
|          6|  India|          IN|       India|
|          7|  India|          IN|       India|
|          8|  India|          IN|       India|
|          9|  India|          IN|       India|
|         10|  India|          IN|       India|
|         11|  India|          IN|       India|
|         12|  India|          IN|       India|
|         13|  India|          IN|       India|
|         14|  India|          IN|       India|
|         15|  India|          IN|       India|
|         16|  India|          IN|       India|
|         17|  India|          IN|       India|
|         18|  India|          IN|      

Assignment 14 — cache() and persist()
Use the orders dataset.
Questions
Explain:
• What is lazy evaluation?
• Why does cache help?
• When should we cache?
• When should we NOT cache?
• Difference between cache() and persist().
• What happens when the cached data does not fit in memory?

In [ ]:
from pyspark.sql import SparkSession
from pyspark.storagelevel import StorageLevel
from pyspark.sql.functions import col, sum

spark = SparkSession.builder.appName("Assignment 14 - Cache and Persist").master("local[*]").getOrCreate()

orders = spark.read.csv("orders.csv",header=True,inferSchema=True)

# Create a filtered DataFrame
completed_orders = orders.filter(col("order_status") == "Completed")

# 1. Cache the DataFrame
completed_orders.cache()

# First action - data is actually calculated and cached here
print("First count:")
print(completed_orders.count())

# Second action - Spark can use cached data
print("Second count:")
print(completed_orders.count())

# Another action using cached data
completed_orders.groupBy("shipping_state").agg(sum("total_amount").alias("revenue")).show()

# Check storage level
print("Storage level:")
print(completed_orders.storageLevel)


# 2. Unpersist when cache is no longer required
completed_orders.unpersist()


# 3. Using persist()
completed_orders.persist(StorageLevel.MEMORY_AND_DISK)

print("Persist count:")
print(completed_orders.count())

completed_orders.groupBy("payment_method").count().show()

print("Storage level:")
print(completed_orders.storageLevel)


# Remove persisted data
completed_orders.unpersist()

First count:
51
Second count:
51
+--------------+-----------------+
|shipping_state|          revenue|
+--------------+-----------------+
|     Karnataka|41103.96000000001|
|        Kerala|         55401.78|
|    Tamil Nadu|         39967.54|
|Andhra Pradesh|         52114.21|
|         Delhi|          38840.5|
|   Maharashtra|        127589.19|
|     Telangana|         61770.79|
+--------------+-----------------+

Storage level:
Disk Memory Deserialized 1x Replicated
Persist count:
51
+--------------+-----+
|payment_method|count|
+--------------+-----+
|    NetBanking|   12|
|          Card|   14|
|          Cash|   15|
|           UPI|   10|
+--------------+-----+

Storage level:
Disk Memory Serialized 1x Replicated


1. What is Lazy Evaluation?

Lazy evaluation means Spark does not immediately execute transformations.

For example:

completed_orders = orders.filter(col("order_status") == "Completed")
Spark does not immediately read and filter all the data.
It simply creates a logical execution plan.
The actual computation happens when we perform an action.

Examples of actions:

count()
show()
collect()
write()

So:

filter()
   ↓
Transformation
   ↓
No immediate execution
   ↓
count()
   ↓
Action
   ↓
Spark executes the plan
Example
completed_orders = orders.filter(col("order_status") == "Completed")
print(completed_orders.count())
The filter() is lazy.
The count() triggers execution.

2. Why does cache() help?

Suppose we use the same DataFrame several times:
completed_orders = orders.filter(col("order_status") == "Completed")
Then:
completed_orders.count()
and later:
completed_orders.groupBy("shipping_state").count().show()
Without caching, Spark may need to recalculate the filtered DataFrame for each separate action.
With:
completed_orders.cache()
Spark keeps the computed data in memory after the first action.
Conceptually:

Without cache:

orders
  ↓
filter
  ↓
count

orders
  ↓
filter
  ↓
groupBy

The filter can be recalculated.

With cache:

orders
  ↓
filter
  ↓
CACHE
 ↙   ↘
count  groupBy

The cached result can be reused.

3. When should we cache?

Cache when:

The DataFrame is used multiple times

For example:

completed_orders.cache()

completed_orders.count()

completed_orders.groupBy("shipping_state").count().show()

completed_orders.groupBy("payment_method").count().show()

The same completed_orders DataFrame is being reused.

The computation is expensive

For example, a DataFrame involving:

multiple joins
aggregations
complicated transformations
large calculations

If the result is reused several times, caching can save recomputation.

Interactive analysis

If you're repeatedly querying the same DataFrame while exploring data, caching can improve response time after the first computation.

4. When should we NOT cache?

Don't automatically cache every DataFrame.

Small DataFrame used only once

For example:

orders.filter(
    col("total_amount") > 5000
).show()

If you use it only once, caching usually provides little benefit.

Very large DataFrame

If the DataFrame is huge and you don't have enough memory, caching can cause memory pressure.

DataFrame isn't reused

For example:

orders.select("order_id", "total_amount").show()

There is no reason to cache it if you aren't going to use it again.

Too many cached DataFrames

Caching many DataFrames can consume executor memory unnecessarily.

5. Difference between cache() and persist()
cache()
completed_orders.cache()

cache() uses Spark's default caching storage level.

For DataFrames, this is effectively:

MEMORY_AND_DISK

So Spark can keep the data in memory and use disk if necessary.

persist()

persist() allows you to specify the storage level.

For example:

completed_orders.persist(StorageLevel.MEMORY_AND_DISK)

You can choose different storage strategies.

Common examples:

StorageLevel.MEMORY_ONLY
StorageLevel.MEMORY_AND_DISK
StorageLevel.DISK_ONLY

So the basic difference is:

cache()	persist()
Uses default storage level	Allows you to specify storage level
Simple	More control
df.cache()	df.persist(...)

Think:

cache()   = simple/default
persist() = choose storage strategy
6. What happens when cached data doesn't fit in memory?

This depends on the storage level.

With:

df.persist(StorageLevel.MEMORY_AND_DISK)

Spark tries to keep the data in memory.

If it doesn't fit completely in memory, Spark can store the remaining partitions on disk.

Conceptually:

Data
 ↓
Memory
 ↓
Doesn't fit completely
 ↓
Remaining partitions
 ↓
Disk

Therefore:

MEMORY_AND_DISK

means:

Keep as much as possible in memory and spill/store the rest on disk.

If you use:

StorageLevel.MEMORY_ONLY

and the data doesn't fit, Spark does not store the excess partitions on disk. Those partitions can be recomputed when needed.

Very important: Cache is lazy too

This is a common interview question.

When you write:

completed_orders.cache()

Spark does not immediately load everything into memory.

cache() is also lazy.

The actual caching happens when an action runs:

completed_orders.cache()

completed_orders.count()

The count() triggers computation, and Spark stores the resulting partitions according to the caching storage level.

unpersist()

When you're finished with the cached DataFrame:

completed_orders.unpersist()

This tells Spark that the cached data is no longer needed.

So the typical pattern is:

df.cache()

# actions using df
df.count()
df.show()

# finished
df.unpersist()

MIXED TOPIC ASSIGNMENTS
E-commerce Sales Analysis
Business Requirement
The company wants to understand its sales performance.
Using:
customers
orders
order_items
products
produce a sales analytics dataset.
Requirements
1. Identify completed orders.
2. Calculate the net order amount.
3. Calculate total revenue by product category.
4. Find the top 5 products in every category.
5. Calculate customer lifetime spending.
6. Identify the highest-value customers.
7. Find customers who have never placed an order.
8. Find products that have never been sold.
9. Calculate monthly revenue.
10.Calculate month-over-month revenue change.
11. Find each customer's first and latest order.
12.Calculate average order value by customer.
13.Create customer segments:
> ₹100,000 → Platinum
₹50,000–₹100,000 → Gold
₹10,000–₹50,000 → Silver
< ₹10,000 → Bronze

In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("E-commerce Sales Analysis").master("local[*]").getOrCreate()

customers = spark.read.csv("customers.csv",header=True,inferSchema=True)
orders = spark.read.csv("orders.csv",header=True,inferSchema=True)

order_items = spark.read.csv("order_items.csv",header=True,inferSchema=True)

products = spark.read.csv("products.csv",header=True,inferSchema=True)

# 1. Identify completed orders
completed_orders = orders.filter(col("order_status") == "Completed")
completed_orders.show(5)

# 2. Calculate net order amount
completed_orders = completed_orders.withColumn("net_order_amount",col("total_amount") - col("discount_amount"))
completed_orders.show(5)

# 3. Total revenue by product category
category_revenue = completed_orders.join(order_items,"order_id","inner").join(products,"product_id","inner").groupBy("category").agg(
    sum("item_amount").alias("total_revenue")).orderBy(col("total_revenue").desc())
category_revenue.show(5)

# 4. Top 5 products in every category
product_revenue = completed_orders.join(order_items,"order_id","inner").join(products,"product_id","inner").groupBy(
    "category","product_id","product_name").agg(sum("item_amount").alias("revenue"))
category_window = Window.partitionBy("category").orderBy(col("revenue").desc())
top_5_products = product_revenue.withColumn("category_rank",row_number().over(category_window)).filter(col("category_rank") <= 5)
top_5_products.show(5)

# 5. Customer lifetime spending
customer_spending = completed_orders.groupBy("customer_id").agg(
    sum("net_order_amount").alias("lifetime_spending"))
customer_spending.show(5)

# 6. Highest-value customers
highest_value_customers = customer_spending.orderBy(col("lifetime_spending").desc())
highest_value_customers.show(5)

# 7. Customers who never placed an order
customers_without_orders = customers.join(orders,"customer_id","left_anti")
customers_without_orders.show(5)

# 8. Products never sold
sold_products = order_items.select("product_id").distinct()
products_never_sold = products.join(sold_products,"product_id","left_anti")
products_never_sold.show(5)

# 9. Monthly revenue
monthly_revenue = completed_orders.withColumn("month",date_format("order_date", "yyyy-MM")).groupBy("month").agg(
    sum("net_order_amount").alias("revenue")).orderBy("month")
monthly_revenue.show(5)

# 10. Month-over-month revenue change
monthly_window = Window.orderBy("month")
monthly_revenue = monthly_revenue.withColumn("previous_month_revenue",lag("revenue").over(monthly_window)).withColumn(
    "revenue_change",col("revenue") - col("previous_month_revenue"))
monthly_revenue.show(5)

# 11. First and latest order for each customer
customer_order_window = Window.partitionBy("customer_id").orderBy(col("order_date").asc())
first_orders = orders.withColumn("row_num",row_number().over(customer_order_window)).filter(
    col("row_num") == 1).select("customer_id",
    col("order_date").alias("first_order_date"))
latest_order_window = Window.partitionBy("customer_id").orderBy(
    col("order_date").desc())
latest_orders = orders.withColumn("row_num",row_number().over(latest_order_window)).filter(
    col("row_num") == 1).select("customer_id",
    col("order_date").alias("latest_order_date"))
first_latest_orders = first_orders.join(latest_orders,"customer_id","inner")
first_latest_orders.show(5)

# 12. Average order value by customer
average_order_value = orders.groupBy("customer_id").agg(
    avg("total_amount").alias("average_order_value"))
average_order_value.show(5)


# 13. Customer segments
customer_segments = customer_spending.withColumn("customer_segment",
    when(col("lifetime_spending") > 100000, "Platinum")
    .when(col("lifetime_spending") >= 50000, "Gold")
    .when(col("lifetime_spending") >= 10000, "Silver")
    .otherwise("Bronze"))
customer_segments.show(5)

# Save monthly revenue
monthly_revenue.write.mode("overwrite").parquet("output/mixed_sales_monthly_revenue")

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|    1002|         36|2025-11-03|   Completed|    NetBanking|       Mumbai|   Maharashtra|    13996.81|         207.17|        46.55|
|    1003|          7|2026-07-21|   Completed|    NetBanking|        Kochi|        Kerala|     2585.08|         218.51|        66.83|
|    1004|         41|2025-09-28|   Completed|          Card|    Bengaluru|     Karnataka|     7729.51|         502.24|       165.89|
|    1006|         29|2026-07-19|   Completed|          Card|         Pune|   Maharashtra|    10673.31|        1121.81|       237.36|
|    1008|          3|2025-11-12|   Completed|    NetBanking| 

Customer 360
Business Requirement
Build a single customer-level dataset.
The final output should contain:
customer_id
customer_name
email
city
registration_date
total_orders
completed_orders
cancelled_orders
total_spending
average_order_value
first_order_date
last_order_date
days_since_last_order
highest_order_value
customer_rank
customer_segment
Additional requirements
1. Customers with no orders must still appear.
2. Handle NULL values.
3. Standardize customer names.
4. Extract email domain.
5. Calculate customer age.
6. Rank customers by total spending.
7. Write the final output as partitioned Parquet.

In [19]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Customer 360").master("local[*]").getOrCreate()

customers = spark.read.csv("customers.csv",header=True,inferSchema=True)
orders = spark.read.csv("orders.csv",header=True,inferSchema=True)

# Order statistics for every customer
order_stats = orders.groupBy("customer_id").agg(
    count("order_id").alias("total_orders"),
    sum(when(col("order_status") == "Completed", 1).otherwise(0)).alias("completed_orders"),
    sum(when(col("order_status") == "Cancelled", 1).otherwise(0)).alias("cancelled_orders"),
    sum(when(
            col("order_status") == "Completed",
            col("total_amount")).otherwise(0)).alias("total_spending"),
    avg("total_amount").alias("average_order_value"),
    min("order_date").alias("first_order_date"),
    max("order_date").alias("last_order_date"),
    max("total_amount").alias("highest_order_value"))

# Join customers with order statistics
customer_360 = customers.join(order_stats,"customer_id","left")

# Handle NULL values for customers without orders
customer_360 = customer_360.withColumn("total_orders",
    coalesce(col("total_orders"), col("customer_id") * 0)).withColumn("completed_orders",
    coalesce(col("completed_orders"), col("customer_id") * 0)).withColumn("cancelled_orders",
    coalesce(col("cancelled_orders"), col("customer_id") * 0)).withColumn("total_spending",
    coalesce(col("total_spending"), col("customer_id") * 0)).withColumn("average_order_value",
    coalesce(col("average_order_value"), col("customer_id") * 0)).withColumn("highest_order_value",
    coalesce(col("highest_order_value"), col("customer_id") * 0))


# Standardize customer name
customer_360 = customer_360.withColumn("customer_name",concat_ws(" ",initcap(col("first_name")),initcap(col("last_name"))))

# Extract email domain
customer_360 = customer_360.withColumn("email_domain",regexp_extract(col("email"),r"@(.+)$", 1))

# Calculate customer age
customer_360 = customer_360.withColumn("customer_age",(datediff(current_date(),col("date_of_birth")) / 365.25).cast("int"))

# Days since last order
customer_360 = customer_360.withColumn("days_since_last_order",when(col("last_order_date").isNull(),None).otherwise(datediff(
            current_date(),col("last_order_date"))))

# Rank customers by total spending
rank_window = Window.orderBy(col("total_spending").desc())
customer_360 = customer_360.withColumn("customer_rank",row_number().over(rank_window))

# Customer segments
customer_360 = customer_360.withColumn("customer_segment",
    when(col("total_spending") > 100000, "Platinum")
    .when(col("total_spending") >= 50000, "Gold")
    .when(col("total_spending") >= 10000, "Silver").otherwise("Bronze"))

# Select final columns
final_customer_360 = customer_360.select(
    "customer_id",
    "customer_name",
    "email",
    "city",
    "registration_date",
    "total_orders",
    "completed_orders",
    "cancelled_orders",
    "total_spending",
    "average_order_value",
    "first_order_date",
    "last_order_date",
    "days_since_last_order",
    "highest_order_value",
    "customer_rank",
    "customer_segment")
final_customer_360.show(5,truncate=False)


# Write as Parquet
final_customer_360.write.mode("overwrite").parquet("output/customer_360")


+-----------+-------------+------------------------+---------+-----------------+------------+----------------+----------------+--------------+-------------------+----------------+---------------+---------------------+-------------------+-------------+----------------+
|customer_id|customer_name|email                   |city     |registration_date|total_orders|completed_orders|cancelled_orders|total_spending|average_order_value|first_order_date|last_order_date|days_since_last_order|highest_order_value|customer_rank|customer_segment|
+-----------+-------------+------------------------+---------+-----------------+------------+----------------+----------------+--------------+-------------------+----------------+---------------+---------------------+-------------------+-------------+----------------+
|20         |Pooja Nair   |pooja.nair20@gmail.com  |Mumbai   |2023-07-18       |2           |2               |0               |25431.47      |12715.735          |2026-03-11      |2026-08-17    

Product Performance
Use:
products
order_items
orders
Requirements
Find:
1. Total quantity sold per product.
2. Total revenue per product.
3. Total profit per product.
4. Average selling price.
5. Number of unique customers.
6. Top 3 products per category.
7. Products with zero sales.
8. Products launched in the last 2 years.
9. Products whose revenue is above their category average.
10. Products whose sales increased month-over-month.
Final output
product_id
product_name
category
brand
quantity_sold
revenue
profit
unique_customers
category_rank
Write the result as Parquet partitioned by:
category

In [21]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Product Performance").master("local[*]").getOrCreate()

products = spark.read.csv("products.csv",header=True,inferSchema=True)
order_items = spark.read.csv("order_items.csv",header=True,inferSchema=True)
orders = spark.read.csv("orders.csv",header=True,inferSchema=True)

# Only completed orders
completed_orders = orders.filter(col("order_status") == "Completed")

# Product sales data
product_sales = completed_orders.join(order_items,"order_id","inner").groupBy(
    "product_id").agg(sum("quantity").alias("quantity_sold"),
    sum("item_amount").alias("revenue"),countDistinct("customer_id").alias("unique_customers"))

# 1, 2, 3, 4, 5
product_performance = products.join(product_sales,"product_id","left")

# Replace NULL sales with zero
product_performance = product_performance.fillna({"quantity_sold": 0,"revenue": 0,"unique_customers": 0})

# 3. Total profit
product_performance = product_performance.withColumn("profit",(col("selling_price") - col("unit_cost"))* col("quantity_sold"))

# 4. Average selling price
# Product table already contains selling_price
product_performance = product_performance.withColumn("average_selling_price", col("selling_price"))

# 6. Top 3 products per category
category_window = Window.partitionBy("category").orderBy(col("revenue").desc())
product_performance = product_performance.withColumn( "category_rank", row_number().over(category_window))
# 7. Products with zero sales
zero_sales = product_performance.filter(col("quantity_sold") == 0)
zero_sales.show()

# 8. Products launched in last 2 years
recent_products = product_performance.filter(col("launch_date") >= add_months(current_date(), -24))
recent_products.show(5)

# 9. Revenue above category average
category_average = product_performance.groupBy("category").agg(
    avg("revenue").alias("category_average_revenue"))
above_category_average = product_performance.join(category_average,"category"
).filter(col("revenue") > col("category_average_revenue"))
above_category_average.show(5)

# 10. Sales increased month-over-month
monthly_product_sales = completed_orders.join(order_items,"order_id","inner").withColumn("sales_month",
    date_format("order_date", "yyyy-MM")).groupBy("product_id","sales_month").agg(
    sum("quantity").alias("monthly_quantity"))
product_month_window = Window.partitionBy("product_id").orderBy("sales_month")
monthly_product_sales = monthly_product_sales.withColumn("previous_month_quantity",
    lag("monthly_quantity").over(product_month_window)).withColumn(
    "sales_increased",col("monthly_quantity") > col("previous_month_quantity"))
sales_increased = monthly_product_sales.filter(col("sales_increased") == True)
sales_increased.show(5)

# Final output
final_product_performance = product_performance.select(
    "product_id",
    "product_name",
    "category",
    "brand",
    "quantity_sold",
    "revenue",
    "profit",
    "unique_customers",
    "category_rank")
final_product_performance.write.mode("overwrite").partitionBy("category").parquet("output/product_performance")
final_product_performance.show(5)


+----------+-----------------+-----------+------------+-----+-----------+---------+-------------+-----------+--------------+--------------+-------------+-------+----------------+------+---------------------+-------------+
|product_id|     product_name|   category|sub_category|brand|supplier_id|unit_cost|selling_price|launch_date|product_rating|stock_quantity|quantity_sold|revenue|unique_customers|profit|average_selling_price|category_rank|
+----------+-----------------+-----------+------------+-----+-----------+---------+-------------+-----------+--------------+--------------+-------------+-------+----------------+------+---------------------+-------------+
|        44|  Prime Fiction 4|      Books|     Fiction|Prime|        144|   8744.0|     10770.02| 2026-01-06|           3.6|           135|            0|    0.0|               0|   0.0|             10770.02|           12|
|        10|Apex Headphones 2|Electronics|  Headphones| Apex|        110|  27779.0|      39045.6| 2025-06-01|   

Monthly Sales Report
Build a monthly business report.
Requirements
For every month calculate:
month
total_orders
completed_orders
cancelled_orders
total_revenue
total_discount
average_order_value
unique_customers
new_customers
Then calculate:
previous_month_revenue
revenue_difference
revenue_growth_percentage
Additional requirements
1. Sort by month.
2. Find the highest-revenue month.
3. Find the lowest-revenue month.
4. Calculate cumulative revenue.
5. Write the report to Parquet.

In [24]:
spark = SparkSession.builder.appName("Product Performance").master("local[*]").getOrCreate()
orders = spark.read.csv("orders.csv",header=True,inferSchema=True)
customers = spark.read.csv("customers.csv",header=True,inferSchema=True)

# Create YYYY-MM
orders_monthly = orders.withColumn("month",date_format("order_date", "yyyy-MM"))

# Monthly report
monthly_report = orders_monthly.groupBy("month").agg(
    count("order_id").alias("total_orders"),
    sum(when(col("order_status") == "Completed",1).otherwise(0)).alias("completed_orders"),
    sum(when(col("order_status") == "Cancelled",1).otherwise(0)).alias("cancelled_orders"),
    sum("total_amount").alias("total_revenue"),
    sum("discount_amount").alias("total_discount"),
    avg("total_amount").alias("average_order_value"),
    countDistinct("customer_id").alias("unique_customers")).orderBy("month")

# New customers by registration month
new_customers = customers.withColumn("month",date_format("registration_date", "yyyy-MM")
).groupBy("month").agg(count("customer_id").alias("new_customers"))

# Add new customers
monthly_report = monthly_report.join(new_customers,"month","left").fillna({"new_customers": 0})

# Previous month revenue
month_window = Window.orderBy("month")
monthly_report = monthly_report.withColumn("previous_month_revenue",lag("total_revenue").over(month_window))

# Revenue difference
monthly_report = monthly_report.withColumn("revenue_difference",col("total_revenue") -col("previous_month_revenue"))


# Revenue growth percentage
monthly_report = monthly_report.withColumn("revenue_growth_percentage",
    when(col("previous_month_revenue").isNull() |
        (col("previous_month_revenue") == 0),None).otherwise(
        (col("revenue_difference") /col("previous_month_revenue")) * 100))

# Cumulative revenue
cumulative_window = Window.orderBy("month").rowsBetween(Window.unboundedPreceding,Window.currentRow)
monthly_report = monthly_report.withColumn( "cumulative_revenue",
    sum("total_revenue").over(cumulative_window))

# Sort by month
monthly_report = monthly_report.orderBy("month")

# Display report
monthly_report.show(5,truncate=False)

# Highest revenue month
highest_revenue_month = monthly_report.orderBy(
    col("total_revenue").desc()).limit(1)
print("Highest Revenue Month:")
highest_revenue_month.show(5)

# Lowest revenue month
lowest_revenue_month = monthly_report.orderBy(
    col("total_revenue").asc()).limit(1)
print("Lowest Revenue Month:")
lowest_revenue_month.show(5)

# Write to Parquet
monthly_report.write.mode("overwrite").parquet("output/monthly_sales_report")


+-------+------------+----------------+----------------+------------------+------------------+-------------------+----------------+-------------+----------------------+------------------+-------------------------+------------------+
|month  |total_orders|completed_orders|cancelled_orders|total_revenue     |total_discount    |average_order_value|unique_customers|new_customers|previous_month_revenue|revenue_difference|revenue_growth_percentage|cumulative_revenue|
+-------+------------+----------------+----------------+------------------+------------------+-------------------+----------------+-------------+----------------------+------------------+-------------------------+------------------+
|2025-01|5           |2               |2               |58216.97          |6692.4800000000005|11643.394          |5               |0            |NULL                  |NULL              |NULL                     |58216.97          |
|2025-02|7           |3               |0               |52520.709999

Payment Reconciliation
Use:
orders
payments
customers
Business Requirement
The finance team wants to identify payment inconsistencies.
Requirements
Find:
1. Orders without payment.
2. Payments without corresponding orders.
3. Orders where payment amount differs from order amount.
4. Failed payments.
5. Refunded payments.
6. Customers with multiple failed payments.
7. Total successful payment amount per month.
8. Payment success rate by payment method.
9. Highest-value failed transactions.
10.Customers with more than 3 failed payments.
Final output
Create:
order_id
customer_id
order_amount
payment_amount
payment_status
payment_difference
reconciliation_status



In [25]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Payment Reconciliation").master("local[*]").getOrCreate()

orders = spark.read.csv("orders.csv",header=True,inferSchema=True)
payments = spark.read.csv("payments.csv",header=True,inferSchema=True)
customers = spark.read.csv("customers.csv",header=True,inferSchema=True)

# 1. Orders without payment
orders_without_payment = orders.join(payments,"order_id","left_anti")
print("Orders without payment:")
orders_without_payment.show(5)

# 2. Payments without corresponding orders
payments_without_order = payments.join(orders,"order_id","left_anti")
print("Payments without corresponding orders:")
payments_without_order.show(5)

# 3. Orders where payment amount differs from order amount
amount_difference = orders.join(payments,"order_id","inner")\
                        .withColumn("payment_difference",col("payment_amount") - col("total_amount")).filter(col("payment_difference") != 0)
print("Orders where payment amount differs:")
amount_difference.show(5)

# 4. Failed payments
failed_payments = payments.filter(col("payment_status") == "Failed")
print("Failed payments:")
failed_payments.show(5)

# 5. Refunded payments
refunded_payments = payments.filter(col("payment_status") == "Refunded")
print("Refunded payments:")
refunded_payments.show(5)

# 6. Customers with multiple failed payments
failed_customer_counts = orders.join(failed_payments,"order_id","inner")\
                                .groupBy("customer_id").agg(count("payment_id").alias("failed_payment_count"))\
                                .filter(col("failed_payment_count") > 1)
print("Customers with multiple failed payments:")
failed_customer_counts.show(5)

# 7. Total successful payment amount per month
successful_monthly_payments = payments.filter(col("payment_status") == "Success")\
                                        .withColumn("month",date_format("payment_date","yyyy-MM"))\
                                            .groupBy("month").agg(sum("payment_amount").alias("successful_payment_amount"))\
                                                .orderBy("month")
print("Successful payment amount per month:")
successful_monthly_payments.show(5)

# 8. Payment success rate by payment method
payment_success_rate = payments.groupBy("payment_method")\
                                .agg(count("payment_id").alias("total_payments"),sum(when(col("payment_status") == "Success",1)\
                                    .otherwise(0)).alias("successful_payments"))\
                                        .withColumn("success_rate_percentage",(col("successful_payments") / col("total_payments")) * 100)
print("Payment success rate by payment method:")
payment_success_rate.show(5)

# 9. Highest-value failed transactions
highest_value_failed = orders.join(failed_payments,"order_id","inner").orderBy(col("total_amount").desc())
print("Highest-value failed transactions:")
highest_value_failed.show(5)

# 10. Customers with more than 3 failed payments
customers_more_than_3_failed = orders.join(failed_payments,"order_id","inner")\
    .groupBy("customer_id").agg(count("payment_id").alias("failed_payment_count"))\
        .filter(col("failed_payment_count") > 3)
print("Customers with more than 3 failed payments:")
customers_more_than_3_failed.show(5)

# Final reconciliation dataset
reconciliation = orders.join(payments,"order_id","left")\
    .withColumn("payment_difference",when(col("payment_amount").isNull(),col("total_amount"))\
        .otherwise(col("payment_amount") - col("total_amount")))\
            .withColumn("reconciliation_status",when(col("payment_id").isNull(),"No Payment")\
                .when(col("payment_status") == "Failed","Failed Payment")\
                    .when(col("payment_status") == "Refunded","Refunded")\
                        .when(col("payment_amount") == col("total_amount"),"Matched")\
                            .otherwise("Amount Mismatch")).select(col("order_id"),col("customer_id"),col("total_amount")\
                                .alias("order_amount"),col("payment_amount"),col("payment_status"),col("payment_difference"),col("reconciliation_status"))

print("Final Reconciliation:")
reconciliation.show(10,truncate=False)

# Write final output
reconciliation.write.mode("overwrite").parquet("output/payment_reconciliation")

spark.stop()

Orders without payment:
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|    1003|          7|2026-07-21|   Completed|    NetBanking|        Kochi|        Kerala|     2585.08|         218.51|        66.83|
|    1011|         22|2026-02-26|   Completed|    NetBanking|        Delhi|         Delhi|    12092.86|        1728.13|        56.62|
|    1023|         29|2025-08-07|    Returned|           UPI|         Pune|   Maharashtra|     7814.26|        1587.47|       171.54|
|    1029|         30|2026-06-28|   Cancelled|          Cash|        Delhi|         Delhi|    14822.02|         1190.9|       211.03|
|    1031|         16|2025-10-11|    R

Customer Retention Analysis
Business Requirement
The marketing team wants to understand customer retention.
Using customers and orders:
Requirements
1. Find each customer's first order.
2. Find the second order.
3. Calculate days between first and second orders.
4. Identify repeat customers.
5. Calculate number of orders per customer.
6. Find customers who ordered in consecutive months.
7. Calculate monthly active customers.
8. Calculate customers retained from previous month
9. Calculate retention percentage.
10. Identify customers who haven't ordered in the last 90 days.


In [26]:
spark = SparkSession.builder.appName("Customer Retention Analysis").master("local[*]").getOrCreate()
customers = spark.read.csv("customers.csv",header=True,inferSchema=True)
orders = spark.read.csv("orders.csv",header=True,inferSchema=True)

# 1. Find each customer's first order
order_window = Window.partitionBy("customer_id").orderBy("order_date")
first_orders = orders.withColumn("row_num",row_number().over(order_window)).filter(col("row_num") == 1)
print("First order of each customer:")
first_orders.select("customer_id","order_id","order_date").show(5)

# 2. Find the second order
second_orders = orders.withColumn("row_num",row_number().over(order_window)).filter(col("row_num") == 2)
print("Second order of each customer:")
second_orders.select("customer_id","order_id","order_date").show(5)

# 3. Calculate days between first and second orders
first_order_dates = first_orders.select("customer_id",col("order_date").alias("first_order_date"))
second_order_dates = second_orders.select("customer_id",col("order_date").alias("second_order_date"))
days_between_orders = first_order_dates.join(second_order_dates,"customer_id","inner")\
    .withColumn("days_between",datediff("second_order_date","first_order_date"))
print("Days between first and second orders:")
days_between_orders.show(5)

# 4. Identify repeat customers
repeat_customers = orders.groupBy("customer_id").agg(count("order_id").alias("order_count"))\
    .filter(col("order_count") > 1)
print("Repeat customers:")
repeat_customers.show(5)

# 5. Calculate number of orders per customer
orders_per_customer = orders.groupBy("customer_id").agg(count("order_id").alias("order_count"))\
    .orderBy(col("order_count").desc())
print("Orders per customer:")
orders_per_customer.show(5)

# 6. Find customers who ordered in consecutive months
customer_months = orders.select("customer_id",date_format("order_date","yyyy-MM").alias("order_month")).distinct()
month_window = Window.partitionBy("customer_id").orderBy("order_month")
consecutive_months = customer_months.withColumn("previous_month",lag("order_month").over(month_window))\
    .filter(col("previous_month").isNotNull())
print("Customers who ordered in consecutive months:")
consecutive_months.show(5)

# 7. Calculate monthly active customers
monthly_active_customers = orders.withColumn("month",date_format("order_date","yyyy-MM"))\
    .groupBy("month").agg(count("customer_id").alias("active_customers")).orderBy("month")
print("Monthly active customers:")
monthly_active_customers.show(5)

# 8. Calculate customers retained from previous month
monthly_customers = orders.withColumn("month",date_format("order_date","yyyy-MM"))\
    .select("month","customer_id").distinct()

previous_month_customers = monthly_customers.withColumn("previous_month",
    lag("month").over(Window.partitionBy("customer_id").orderBy("month")))

retained_customers = previous_month_customers.filter(col("previous_month").isNotNull())\
    .groupBy("month").agg(count("customer_id").alias("retained_customers")).orderBy("month")

print("Customers retained from previous month:")
retained_customers.show(5)

# 9. Calculate retention percentage
retention_report = monthly_active_customers.join(retained_customers,"month","left")\
    .fillna(0).withColumn("retention_percentage",
        (col("retained_customers") / col("active_customers")) * 100)

print("Retention percentage:")
retention_report.show(5)

# 10. Identify customers who haven't ordered in the last 90 days
recent_orders = orders.filter(col("order_date") >= date_sub(current_date(),90))\
    .select("customer_id").distinct()

inactive_customers = customers.join(recent_orders,"customer_id","left_anti")
print("Customers who haven't ordered in the last 90 days:")
inactive_customers.show(5)

# Final retention output
retention_report.write.mode("overwrite").parquet("output/customer_retention_report")


First order of each customer:
+-----------+--------+----------+
|customer_id|order_id|order_date|
+-----------+--------+----------+
|          1|    1022|2025-11-08|
|          2|    1056|2025-04-06|
|          3|    1092|2025-05-10|
|          4|    1098|2025-10-28|
|          5|    1032|2025-10-20|
+-----------+--------+----------+
only showing top 5 rows

Second order of each customer:
+-----------+--------+----------+
|customer_id|order_id|order_date|
+-----------+--------+----------+
|          1|    1018|2026-06-17|
|          3|    1008|2025-11-12|
|          5|    1019|2025-12-05|
|          6|    1041|2026-04-27|
|          7|    1049|2026-03-20|
+-----------+--------+----------+
only showing top 5 rows

Days between first and second orders:
+-----------+----------------+-----------------+------------+
|customer_id|first_order_date|second_order_date|days_between|
+-----------+----------------+-----------------+------------+
|          1|      2025-11-08|       2026-06-17|     

Top-N Analysis
Requirement
The business wants Top-N reports.
Calculate:
Top 10 customers
Based on:
total spending
Top 5 products
Within every:
category
Top 3 states
Based on:
revenue
Top 5 orders
For every customer.
Additional
Calculate the percentage contribution of each product to its category revenue.

In [ ]:
spark = SparkSession.builder.appName("Top-N Analysis").master("local[*]").getOrCreate()
customers = spark.read.csv("customers.csv",header=True,inferSchema=True)
orders = spark.read.csv("orders.csv",header=True,inferSchema=True)
order_items = spark.read.csv("order_items.csv",header=True,inferSchema=True)
products = spark.read.csv("products.csv",header=True,inferSchema=True)

# 1. Top 10 customers based on total spending
customer_spending = orders.groupBy("customer_id").agg(sum("total_amount").alias("total_spending"))\
    .orderBy(col("total_spending").desc()).limit(10)

print("Top 10 customers based on total spending:")
customer_spending.show()

# 2. Top 5 products within every category
product_revenue = orders.join(order_items,"order_id","inner")\
    .join(products,"product_id","inner")\
    .groupBy("category","product_id","product_name")\
    .agg(sum("item_amount").alias("product_revenue"))

category_window = Window.partitionBy("category").orderBy(col("product_revenue").desc())

top_5_products = product_revenue.withColumn("product_rank",row_number().over(category_window))\
    .filter(col("product_rank") <= 5)

print("Top 5 products within every category:")
top_5_products.show()

# 3. Top 3 states based on revenue
top_3_states = orders.groupBy("shipping_state").agg(sum("total_amount").alias("revenue"))\
    .orderBy(col("revenue").desc()).limit(3)

print("Top 3 states based on revenue:")
top_3_states.show()

# 4. Top 5 orders for every customer
customer_order_window = Window.partitionBy("customer_id").orderBy(col("total_amount").desc())

top_5_orders = orders.withColumn("order_rank",row_number().over(customer_order_window))\
    .filter(col("order_rank") <= 5)

print("Top 5 orders for every customer:")
top_5_orders.select("customer_id","order_id","total_amount","order_rank").show()

# 5. Percentage contribution of each product to its category revenue
category_revenue = product_revenue.groupBy("category")\
    .agg(sum("product_revenue").alias("category_revenue"))

product_contribution = product_revenue.join(category_revenue,"category","inner")\
    .withColumn("percentage_contribution",
        round((col("product_revenue") / col("category_revenue")) * 100,2))

print("Product percentage contribution to category revenue:")
product_contribution.select("category","product_id","product_name","product_revenue","category_revenue","percentage_contribution").show()

# Final outputs
customer_spending.write.mode("overwrite").parquet("output/top_10_customers")
top_5_products.write.mode("overwrite").parquet("output/top_5_products_category")
top_3_states.write.mode("overwrite").parquet("output/top_3_states")
top_5_orders.write.mode("overwrite").parquet("output/top_5_orders_customer")
product_contribution.write.mode("overwrite").parquet("output/product_category_contribution")

Top 10 customers based on total spending:
+-----------+------------------+
|customer_id|    total_spending|
+-----------+------------------+
|         30| 76160.45000000001|
|         16| 62607.06999999999|
|         36|          50962.46|
|         19|           46037.9|
|         28|          37560.33|
|         31|33421.799999999996|
|         41|33012.369999999995|
|         45|30077.850000000002|
|         23|29447.550000000003|
|         29|          29129.61|
+-----------+------------------+

Top 5 products within every category:
+-----------+----------+------------------+------------------+------------+
|   category|product_id|      product_name|   product_revenue|product_rank|
+-----------+----------+------------------+------------------+------------+
|      Books|        39|Urban Technology 3|         606879.67|           1|
|      Books|        41|    Nova Fiction 1|         567803.83|           2|
|      Books|        46|  Apex Education 2| 539526.8999999999|           3|
|

Data Quality Assignment
This one is very useful for teaching real-world PySpark.
1. Identify NULL records.
2. Identify duplicates.
3. Identify invalid emails.
4. Identify negative amounts.
5. Identify invalid dates.
6. Standardize text fields.
7. Remove invalid records.
8. Create a data-quality status column.
9. Generate a rejected-record dataset.
10. Generate a cleaned dataset.
Final outputs:
clean_orders/
rejected_orders/
data_quality_report/

In [30]:
spark = SparkSession.builder.appName("Data Quality Analysis").master("local[*]").getOrCreate()
orders = spark.read.csv("orders.csv",header=True,inferSchema=True)
print("Orders columns:")
print(orders.columns)

# 1. Identify NULL records
null_records = orders.filter(
    col("order_id").isNull() |
    col("customer_id").isNull() |
    col("order_date").isNull() |
    col("order_status").isNull() |
    col("payment_method").isNull() |
    col("shipping_city").isNull() |
    col("shipping_state").isNull() |
    col("total_amount").isNull())
print("NULL records:")
null_records.show(5)

# 2. Identify duplicate records
duplicate_records = orders.groupBy("order_id").agg(
    count("*").alias("order_count")).filter(col("order_count") > 1)
print("Duplicate records:")
duplicate_records.show(5)

# 3. Identify invalid payment methods
invalid_payment_methods = orders.filter(~col("payment_method").isin(
        "Credit Card",
        "Debit Card",
        "UPI",
        "Cash",
        "Net Banking"))
print("Invalid payment method records:")
invalid_payment_methods.show(5)

# 4. Identify negative amounts
negative_amounts = orders.filter(col("total_amount") < 0)
print("Negative amount records:")
negative_amounts.show(5)

# 5. Identify invalid dates
orders_with_date = orders.withColumn("valid_order_date",
    to_date(col("order_date"),"yyyy-MM-dd"))

invalid_dates = orders_with_date.filter(col("valid_order_date").isNull())
print("Invalid date records:")
invalid_dates.show(5)

# 6. Standardize text fields
standardized_orders = orders.withColumn("order_status",
    upper(trim(col("order_status"))))\
        .withColumn("payment_method",
            initcap(trim(col("payment_method"))))\
        .withColumn("shipping_city",
            initcap(trim(col("shipping_city"))))\
        .withColumn("shipping_state",initcap(trim(col("shipping_state"))))
print("Standardized records:")
standardized_orders.show(5)

# 7. Create data quality status
quality_orders = standardized_orders.withColumn("data_quality_status",when(
        col("order_id").isNull() |
        col("customer_id").isNull() |
        col("order_date").isNull() |
        col("order_status").isNull() |
        col("payment_method").isNull() |
        col("shipping_city").isNull() |
        col("shipping_state").isNull() |
        col("total_amount").isNull() |
        (col("total_amount") < 0),
        "REJECTED").otherwise("VALID"))
print("Data quality status:")
quality_orders.select("order_id","customer_id","total_amount","data_quality_status").show(5)

# 8. Generate rejected-record dataset
rejected_orders = quality_orders.filter(
    col("data_quality_status") == "REJECTED")
print("Rejected records:")
rejected_orders.show(5)

# 9. Generate cleaned dataset
clean_orders = quality_orders.filter(
    col("data_quality_status") == "VALID")
print("Cleaned records:")
clean_orders.show(5)

# 10. Generate data quality report
total_records = orders.count()
null_count = null_records.count()
duplicate_count = duplicate_records.count()
invalid_payment_count = invalid_payment_methods.count()
negative_amount_count = negative_amounts.count()
invalid_date_count = invalid_dates.count()
clean_count = clean_orders.count()
rejected_count = rejected_orders.count()

data_quality_report = spark.createDataFrame([
    ("Total Records",total_records),
    ("NULL Records",null_count),
    ("Duplicate Records",duplicate_count),
    ("Invalid Payment Method Records",invalid_payment_count),
    ("Negative Amount Records",negative_amount_count),
    ("Invalid Date Records",invalid_date_count),
    ("Clean Records",clean_count),
    ("Rejected Records",rejected_count)
],["quality_check","record_count"])

print("Data Quality Report:")
data_quality_report.show()

# 11. Save cleaned dataset
clean_orders.write.mode("overwrite").parquet("clean_orders")

# 12. Save rejected dataset
rejected_orders.write.mode("overwrite").parquet("rejected_orders")

# 13. Save data quality report
data_quality_report.write.mode("overwrite").parquet("data_quality_report")

Orders columns:
['order_id', 'customer_id', 'order_date', 'order_status', 'payment_method', 'shipping_city', 'shipping_state', 'total_amount', 'discount_amount', 'shipping_cost']
NULL records:
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_city|shipping_state|total_amount|discount_amount|shipping_cost|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------------+

Duplicate records:
+--------+-----------+
|order_id|order_count|
+--------+-----------+
+--------+-----------+

Invalid payment method records:
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+---------------+-------